In [1]:
import pandas as p

In [2]:
fg26 = p.read_table('/home/matej/ysh1/ysh1_fg_t26_170316.txt')

In [3]:
ts30 = p.read_table('/home/matej/ysh1/ysh1_ts_t26_170316.txt')

In [4]:
scores = p.concat([fg26, ts30])

In [5]:
g = scores.groupby(list(scores.columns[:4]))

In [6]:
def foo(grp):
    if grp.shape[0] > 1:
        return grp[grp['Arraytype/Temp'] == 'DMA26'].iloc[0]
    return grp.iloc[0]
consolidated = g.apply(foo).reset_index(drop=True)

In [51]:
consolidated['queryid'] = consolidated['Query Strain ID'].apply(lambda x: x.split('_')[-1])

In [7]:
tsq3092 = consolidated.loc[consolidated['Query Strain ID'] == 'YLR277C_tsq3092']
tsq3093 = consolidated.loc[consolidated['Query Strain ID'] == 'YLR277C_tsq3093']

In [9]:
ds = Dataset.objects.get(pk=19)

In [13]:
score_axis = []
for a in ds.arrays.all():
    score_axis.append(a.boonelab_id)

In [19]:
tsq3092.index = tsq3092['Array Strain ID'].apply(lambda x: x.split('_')[-1])
tsq3093.index = tsq3093['Array Strain ID'].apply(lambda x: x.split('_')[-1])

In [23]:
tsq3092 = tsq3092.reindex(score_axis)
tsq3093 = tsq3093.reindex(score_axis)

In [29]:
data = StrainData.objects.get(pk=115417)
data.scores = tsq3092['Genetic interaction score (ε)']
data.pvalues = tsq3092['P-value']
data.save()

In [30]:
data = StrainData.objects.get(pk=115418)
data.scores = tsq3093['Genetic interaction score (ε)']
data.pvalues = tsq3093['P-value']
data.save()

In [35]:
livestraindatas = {x.strain.boonelab_id: x for x in StrainData.objects.filter(dataset=19).defer('scores', 'pvalues', 'correlations')}

In [53]:
g = consolidated.groupby('Array Strain ID')

In [ ]:
for grp, grpdata in g:
    strainid = grp.split('_')[-1]
    if strainid not in livestraindatas:
        continue
    
    grpdata = grpdata.set_index('queryid').reindex(['tsq3092', 'tsq3093'])
    print "UPDATE base_straindata SET pvalues[5356:5357] = '{%s}', scores[5356:5357] = '{%s}' WHERE id = %s;" % (
        ','.join(map(str, grpdata['P-value'])),
        ','.join(map(str, grpdata['Genetic interaction score (ε)'])),
        livestraindatas[strainid].id)
    

In [69]:
xx = StrainData.objects.get(id=110733)

In [70]:
xxscores = xx.scores
xxpval = xx.pvalues
xxcorrelations = xx.correlations

In [72]:
xx.scores[-10:]

[0.022, nan, nan, nan, nan, -0.109, nan, -0.0749, nan, nan]

In [73]:
queries = list(ds.queries.all())
arrays = list(ds.arrays.all())
correlations = list(ds.correlation_axis.all())

In [79]:
yshcorr = p.read_table('/home/matej/ysh1/ysh1_corr.tsv', index_col=0)

In [80]:
yshcorr

,aac1,aac3,aad10,aad14,aad15,aad16,aad3,aad4,aad6,aah1,...,zpr1-1,zrc1,zrg17,zrg8,zrt1,zrt2,zrt3,zta1,zuo1,zwf1
ysh1-5003,0.011913,-0.029085,0.024986,0.007301,-0.036707,0.024384,-0.014819,0.038231,0.014366,NaN,...,-0.017070,-0.022149,-0.067811,0.033882,-0.019110,0.040079,-0.022637,0.000830,-0.013551,0.009430
ysh1-5004,-0.028935,-0.026827,0.030942,0.003391,-0.033443,0.035107,-0.018646,0.001139,0.009762,NaN,...,-0.010282,-0.010072,-0.016759,0.020477,0.014712,0.032842,-0.009345,0.013945,-0.033709,-0.003292


In [86]:
corrlabelaxis = [c.label().lower() for c in correlations]

In [104]:
yshcorr.reindex(columns=corrlabelaxis)

,tfc3-g349e,vps8,fun14,spo7,mdm10,swc3,cys3,dep1,syn8,ntg1,...,vph1-supp1,rmi1-supp1,kap120-supp1,ypr050c,pre2-1-supp1,ypr123c-supp1,rps23b-supp1,bet2-1-supp1,ysh1-5003,ysh1-5004
ysh1-5003,-0.016658,-0.00804,0.013222,0.053324,0.061052,0.013951,0.000583,0.052090,0.058821,-0.040009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.786611
ysh1-5004,-0.031524,-0.01928,0.028989,0.047423,0.043562,0.058177,-0.036000,0.085819,0.004460,-0.037740,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.786611,NaN


In [102]:
newaxis = []
for x in yshcorr.columns:
    if x not in corrlabelaxis:
        newaxis.append(genes[x])
    else:
        newaxis.append(x)

In [103]:
yshcorr.columns = newaxis

In [96]:
genes = {g.orf.lower(): g.name and g.name.lower() for g in Gene.objects.all()}

In [99]:
genes['aim1'] = 'bol3'
genes['fra2'] = 'bol2'

In [106]:
data = StrainData.objects.get(pk=115417)
data.correlations = yshcorr.loc['ysh1-5003']
data.save()

In [107]:
data = StrainData.objects.get(pk=115418)
data.correlations = yshcorr.loc['ysh1-5004']
data.save()

In [119]:
yshcorr.loc[:, 'clp1-5001']

ysh1-5003    0.220307
ysh1-5004    0.220096
Name: clp1-5001, dtype: float64

In [114]:
livestrainlabels = {x.strain.label().lower(): x for x in StrainData.objects.filter(dataset=19).select_related('strain__gene').defer('scores', 'pvalues', 'correlations')}

In [ ]:
for lbl in yshcorr.columns:
#     strainid = grp.split('_')[-1]
#     if strainid not in livestraindatas:
#         continue
    
    print "UPDATE base_straindata SET correlations[6231:6232] = '{%s}' WHERE id = %s;" % (
        ','.join(map(str, yshcorr[lbl])),
        livestrainlabels[lbl].id)
    
#     break

In [120]:
import json

In [129]:
nodes = json.load(open('/home/matej/vizbkp/nodes.json'))

In [130]:
nodes = {n['label']: n for n in nodes['nodes']}

In [132]:
nodes['ysh1-5003']
# 6231, 6232

{u'alel': u'ysh1-5003',
 u'aliases': [u'cleavagepolyadenylationfactorsubunitYSH1', u'BRR5'],
 u'id': 6231,
 u'isdu': False,
 u'isnf': False,
 u'label': u'ysh1-5003',
 u'name': u'YSH1',
 u'orf': u'YLR277C'}

In [137]:
yshcorr.sort_values('ysh1-5004', axis=1, ascending=False)

,ysh1-5003,clp1-5001,pcf11-2,mpe1-5005,pcf11-ts2,yth1-5005,cft2-5001,rna15-58,alf1,fcp1-1-supp3,...,ypr150w,ypr170c,ypr195c,ypr196w,ypr197c,ypt35,yrr1,ysh1-5004,zeo1,zip1
ysh1-5003,NaN,0.220307,0.153292,0.222003,0.146728,0.147268,0.180563,0.172431,0.082583,0.108907,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.786611,NaN,NaN
ysh1-5004,0.786611,0.220096,0.209540,0.205940,0.189762,0.188144,0.186482,0.162273,0.150853,0.142767,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
